# Task 07: ranker dataset and fold-specific features

Read-only EDA of the materialized task07 artifact. Candidate models are never fitted or invoked here; canonical labels are not used for feature selection.

In [ ]:
from pathlib import Path
import json
import os
import polars as pl

requested = Path(os.environ.get('TASK07_ARTIFACT', 'artifacts/task07_ranker_dataset_v1'))
artifact = requested if requested.exists() else Path('artifacts/task07_ranker_dataset_smoke_a')
if not artifact.exists():
    raise FileNotFoundError('Run task07 preparation or set TASK07_ARTIFACT')
config = json.loads((artifact / 'config.json').read_text())
metrics = json.loads((artifact / 'metrics.json').read_text())
schema = json.loads((artifact / 'feature_schema.json').read_text())
print({'artifact': artifact.as_posix(), 'mode': config['mode'], 'folds': config['fold_order']})

In [ ]:
pl.DataFrame([
    {'fold': fold, **values}
    for fold, values in metrics['folds'].items()
]).select(
    'fold', 'rows', 'users', 'positive_rows', 'positive_rate',
    'hard_negative_rows', 'training_rows', 'training_positive_rate'
)

In [ ]:
selection_folds = [fold for fold in config['fold_order'] if fold != 'canonical']
eda_fold = selection_folds[-1]
parts = (artifact / 'folds' / eda_fold / 'ranker_data' / 'part-*.parquet').as_posix()
frame = pl.scan_parquet(parts)
source_columns = [name for name in frame.collect_schema().names() if name.startswith('generated_by_')]
eda = frame.select(
    pl.len().alias('rows'),
    pl.col('label').sum().alias('positives'),
    pl.col('is_training_sample').sum().alias('training_rows'),
    *[pl.col(name).sum().alias(name) for name in source_columns],
).collect()
print({'eda_fold': eda_fold, 'column_count': schema['column_count'], 'feature_count': schema['feature_count']})
eda

In [ ]:
fold_manifest = json.loads((artifact / 'folds' / eda_fold / 'fold_manifest.json').read_text())
history = fold_manifest['history_diagnostics']
assert history['max_dt'] < history['cutoff']
assert config['candidate_models_invoked'] is False
print({
    'cutoff': history['cutoff'],
    'max_feature_timestamp': history['max_dt'],
    'sampling': config['negative_sampling'],
    'canonical_isolation': 'canonical excluded from this label EDA',
})